### Obx events mapping
- 0 is analog visual
- 2 is trial start
- 3 is frames
- 4 is left port
- 5 is center port
- 6 is right port

In [1]:
from labdata.schema import (
    EphysRecording,
    File,
    Dataset,
    DatasetEvents,
    StreamSync,
    Session,
    SpikeSorting,
)
import numpy as np
from tqdm import tqdm

/Users/gabriel/lib/ephys/.venv/lib/python3.11/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # requires setuptools<82
[2026-03-13 13:30:58,388][INFO]: DataJoint 0.14.9 connected to rojasbowe@churchland-data.cmojfwfr0b9t.us-west-2.rds.amazonaws.com:3306


subject = "GRB058"
session = "20260224_152424"
sess_query = Session() & f'subject_name = "{subject}"' & f'session_name = "{session}"'

In [ ]:
sessions = (EphysRecording() & SpikeSorting() - 'subject_name = "GRB006"').proj()

for ses in tqdm(sessions):
    if len(DatasetEvents & ses & 'stream_name = "obx"'):
        continue
    paths = (
        File() & (Dataset.DataFiles & ses & 'file_path LIKE "%.obx.%"')
    ).check_if_files_local()[0]
    from spks.spikeglx_utils import load_spikeglx_binary
    from spks.sync import unpack_npix_sync

    dat, meta = load_spikeglx_binary(paths[0])
    onsets, offsets = unpack_npix_sync(dat[:, -2])  # unpack the syncs for the behavior
    clock = unpack_npix_sync(dat[:, -1])[0][6]  # unpack the clock
    srate = meta["sRateHz"]
    # insert the events
    stream = "obx"
    DatasetEvents().insert(
        [dict(ses, stream_name=stream)],
        allow_direct_insert=True,
        skip_duplicates=True,
    )
    digital_events = []
    for o in onsets.keys():
        digital_events.append(
            dict(
                ses,
                stream_name=stream,
                event_name=int(o),
                event_timestamps=(onsets[o] / srate).astype(np.float32),
            )
        )
    digital_events.append(
        dict(
            ses,
            stream_name=stream,
            event_name="sync",
            event_timestamps=(clock / srate).astype(np.float32),
        )
    )
    DatasetEvents.Digital().insert(
        digital_events, allow_direct_insert=True, skip_duplicates=True
    )

    StreamSync().insert(
        [
            dict(
                ses,
                stream_name=s,
                event_name=6,
                clock_dataset=ses["dataset_name"],
                clock_stream="nidq",
                clock_stream_event=6,
            )
            for s in ["imec0"]
        ],
        skip_duplicates=True,
    )
    StreamSync().insert(
        [
            dict(
                ses,
                stream_name="obx",
                event_name="sync",
                clock_dataset=ses["dataset_name"],
                clock_stream="nidq",
                clock_stream_event=6,
            )
        ],
        skip_duplicates=True,
    )

  0%|          | 0/4 [00:00<?, ?it/s]

'imroTbl'


100%|██████████| 4/4 [02:05<00:00, 31.38s/it]


In [11]:
DatasetEvents.Digital() & 'session_name = "20260312_134952"'

subject_name unique mouse id,session_name session identifier,dataset_name,"stream_name which clock is used e.g. btss, nidq, bpod, imecX",event_name,event_timestamps timestamps of the events,event_values event value or count
GRB058,20260312_134952,chipmunk,bpod,sync,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,imec0,6,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,nidq,6,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,0,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,2,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,3,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,4,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,5,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,6,=BLOB=,=BLOB=
GRB058,20260312_134952,ephys_g0,obx,sync,=BLOB=,=BLOB=
